## 第13章 继承和混入

### 1.继承

- **面向对象设计原则**：SOLID原则，又称坚实原则。
    - S：Single Responsibility Principle，单一职责原则。类与函数一样，应只有**一个单一的、明确界定的职责**。
    - O：Open-Closed Principle，开放-关闭原则。一旦类在代码中被大量使用，应**避免以影响其使用方式的方法来修改它**。
    - L：Liskov Substitution Principle，里氏替换原则。使用继承时，必须**能在程序运行中用派生类替换基类，而不改变程序的预期行为**。
    - I：Interface Segregation Principle，接口隔离原则。设计时考虑客户端需求，**不要强迫客户端了解或绕过不会使用的接口部分**。
    - D：Dependency Inversion Principle，依赖倒置原则。相互依赖的类导致大量重复代码，替换/修改时需在各处更改，**解决方案是松耦合设计，包括多态、组合等**。

- **何时使用继承**：核心原则，面向对象设计的决策必须基于被封装的数据。
    - 类是由其组成的数据定义的，继承应该扩展这个定义。**如果两个或多个类需要包含相同类型的数据并提供相同的接口，那么继承很可能是合理的。**
    - Python中很少需要用继承和多态处理不同类型数据，而大部分时候使用**鸭子类型**。
    - 如果仅为要求实现特定接口或扩展复杂接口，应考虑**抽象基类**。
    - 继承常见的反模式：
        - 上帝类：缺乏单一明确职责，存储或提供大量共享资源访问。替换方法：使用类属性存储需要在对象间可变共享的内容。
        - 桩类：包含很少甚至没有数据的类，仅为了最小化重复代码。替换方法：使用模块中的普通函数，或采用组合。

In [ ]:
# 基类
class Collection:
    def __init__(self, title, page_start, length=1):
        self.title = title
        self.page_start = page_start
        self.page_end = page_start + length - 1
        self.items = []

    def __str__(self):
        return self.title

# 子类：继承Collection
class MonthlyLog(Collection):
    def __init__(self, month, year, page_start, length=2):
        super().__init__(f'{month}/{year}', page_start, length)  # 调用基类的构造函数
        self.events = []

    def __str__(self):
        return f'{self.title} (Monthly Log)'

# 子类：继承Collection
class FutureLog(Collection):
    def __init__(self, start_month, page_start):
        super().__init__('Future Log', page_start, 4)  # 调用基类的构造函数
        self.start = start_month
        self.months = [start_month]

log = FutureLog('Jan 2026', 5)
print(log)
monthly = MonthlyLog('April', 2026, 9)
print(monthly)
read = Collection('Books to Read', 17)
print(read)

- **多重继承**：当一个类继承自多个基类时，它获得所有基类的属性和方法。
    - 方法解析顺序：C3 MRO技术，可通过`ClassName.__mro__`属性查看。核心规则是：线性化过程始终寻找下一个最近的祖先，只要该祖先**没有被任何尚未考虑的祖先继承**。
        - 可以显式调用想要的基类上的方法，绕过MRO。必须**显式传递`self`**，因为是在类上调用实例方法，而非在实例上调用。
    - 初始化器问题：使用协同初始化器技术，确保每个基类的初始化器都被调用。
        - 每个初始化器**显式接受自己需要的参数**。
        - 必须接受`**kwargs`(未知关键字参数)。
        - 每个初始化器将自己需要的参数取出后，通过`super().__init__()`将**剩余全部向上传递**。


In [ ]:
# 方法解析顺序
class Food:
    def __str__(self):
        return "Yum, what is it?"

class Pizza(Food):
    def __str__(self):
        return "Piiiizzaaaaaa"

class Sandwich(Food):
    def __str__(self):
        return "Mmm, sammich."

class Calzone(Pizza, Sandwich):  # 多重继承
    pass

calzone = Calzone()
print(calzone)  # Piiiizzaaaaaa

# 方法解析顺序：
# L(Food) = Food, object
# L(Pizza) = Pizza, (Food, object) = Pizza, Food, object
# L(Sandwich) = Sandwich, (Food, object) = Sandwich, Food, object
# L(Calzone) = Calzone, (Pizza, Food, object), (Sandwich, Food, object) = Calzone, Pizza, Sandwich, Food, object

In [ ]:
# 继续时指定基类的顺序很重要！
# L(PizzaSandwich) = PizzaSandwich, (Sandwich, Food, object), (Pizza, Food, object) = PizzaSandwich, Sandwich, Pizza, Food, object
class PizzaSandwich(Sandwich, Pizza):
    pass

# 出错：
# L(CalzonePizzaSandwich) = CalzonePizzaSandwich, (Calzone, Pizza, Sandwich, Food, object), (PizzaSandwich, Sandwich, Pizza, Food, object)
# L(CalzonePizzaSandwich) = CalzonePizzaSandwich, Calzone, PizzaSandwich, (Pizza, Sandwich, Food, object), (Sandwich, Pizza, Food, object)
# 接下来无法解析，Pizza、Sandwich互为冲突。
# 解决办法：调整PizzaSandwich的基类顺序，PizzaSandwich(Pizza, Sandwich):
# class CalzonePizzaSandwich(Calzone, PizzaSandwich):
#     pass

In [ ]:
# 初始化器问题
class Food:
    def __init__(self, name):
        self.name = name

class Pizza(Food):
    def __init__(self, toppings):
        super().__init__("Pizza")
        self.toppings = toppings

class Sandwich(Food):
    def __init__(self, bread, fillings):
        super().__init__("Sandwich")
        self.bread = bread
        self.fillings = fillings

class Calzone(Pizza, Sandwich):
    def __init__(self, toppings):
        super().__init__(toppings)

calzone = Calzone("sausage")
# 出现错误: __init__() missing 1 required positional argument: 'fillings'
# 原因：super().__init__() 调用 Pizza.__init__()，然后 Pizza 中的 super().__init__() 沿 Calzone 的 MRO 调用 Sandwich.__init__()，但传递了错误参数。

In [ ]:
# 协同初始化器
# 1. 每个初始化器显式接受自己需要的参数
# 2. 必须接受 **kwargs（未知关键字参数），因为不可能提前知道通过 super().__init__() 可能传递的所有参数
# 3. 每个初始化器将自己需要的参数取出后，通过 super().__init__() 将剩余全部向上传递
# 4. 为 name 提供默认值，以便直接实例化 Pizza 或 Sandwich 时使用

class Food:
    def __init__(self, name):
        self.name = name

class Pizza(Food):
    def __init__(self, toppings, name='Pizza', **kwargs):
        super().__init__(name=name, **kwargs)
        self.toppings = toppings

class Sandwich(Food):
    def __init__(self, bread, fillings, name='Sandwich', **kwargs):
        super().__init__(name=name, **kwargs)
        self.bread = bread
        self.fillings = fillings

class Calzone(Pizza, Sandwich):
    # 1. 只需一次 super().__init__() 调用，根据 MRO 自动指向 Pizza.__init__()
    # 2. 传递了 MRO 中所有初始化器需要的全部参数
    # 3. 只使用关键字参数，每个参数有唯一名称，确保每个初始化器可获取所需内容
    def __init__(self, toppings):
        super().__init__(toppings=toppings, bread='pizza', fillings=toppings, name='Calzone')

pizza = Pizza(toppings="pepperoni")
sandwich = Sandwich(bread="rye", fillings="swiss")
calzone = Calzone("sausage")

### 2.混入

- **混入**：Mixin，混入是一种特殊的不完整的类，包含你可能想要添加到多个其他类中的功能。
    - 当你需要在多个类之间重用相同的方法时，混入是最佳方式之一。
    - 典型用途：跨类共享日志记录、数据库连接、网络、身份验证等通用方法。
    - 与继承的区别：
        - 混入本质上依赖于一种**恰好利用继承机制的组合形式**。
        - 混入**很少有自己的属性**；相反，它通常依赖于对使用它的类的**属性和方法的期望**。
        - 混入**不能单独使用**。

In [ ]:
import configparser
from pathlib import Path

# 混入类
# SettingsMixin 不是完整的类：缺少初始化器，引用了不属于自己的实例属性 self.settings_section，由任何使用混入的类提供
class SettingsMixin:
    settings_path = Path('res/livesettings.ini')
    config = configparser.ConfigParser()

    def read_setting(self, key):
        self.config.read(self.settings_path)  # 读取配置文件
        try:
            return self.config[self.settings_section][key] # type: ignore  此处的settings_section要到实际使用时再赋值
        except KeyError:
            raise KeyError("Invalid section in settings file.")

# 将SettingsMixin混入到Greeter类中
class Greeter(SettingsMixin):
    def __init__(self, greeting):
        self.settings_section = 'MAGIC'
        self.greeting = greeting

    def __str__(self):
        try:
            name = self.read_setting('UserName')
        except KeyError:
            name = 'user'
        return f'{self.greeting}, {name}!'

greeter = Greeter("Salutations,")
for i in range(3):
    print(greeter)

### 3.本章小结

- **核心知识脉络**

```text
继承与混入
│
├── 一、理论回顾：继承
│   └── is-a 关系，派生类可扩展/覆盖基类
│
├── 二、SOLID 原则 ⭐⭐
│   ├── S：单一职责 → 避免"上帝类"
│   ├── O：开闭原则 → 扩展而非修改
│   ├── L：里氏替换 → 派生类可替换基类，不改变行为
│   ├── I：接口隔离 → 按需拆分接口，不强迫客户端
│   └── D：依赖倒置 → 抽象接口 + 松耦合
│       ├── 多态（继承实现公共接口）
│       └── 组合（控制器类抽象细节）
│
├── 三、何时使用继承
│   ├── Python 鸭子类型优先于继承
│   ├── 继承用于：扩展数据定义（is-a + 相同数据）
│   └── 纯接口需求 → 抽象基类（第14章）
│
├── 四、继承的罪行（三大反模式）
│   ├── 上帝类 → 缺乏单一职责
│   ├── 桩类 → 基于代码复用而非数据
│   └── 为"聪明"而用继承 → 应用组合替代
│
├── 五、Python 基础继承
│   ├── Collection → MonthlyLog / FutureLog 示例
│   ├── super().__init__() 显式调用基类初始化 ⭐
│   ├── 派生类可覆盖 / 新增方法
│   └── isinstance() / issubclass() 检查关系
│
├── 六、多重继承 ⭐⭐
│   ├── 方法解析顺序（MRO）
│   │   ├── 菱形继承问题（死亡钻石）
│   │   └── C3 MRO 线性化算法（逐步推导）
│   ├── 确保一致 MRO：基类顺序必须一致 ⚠️
│   ├── 显式解析：Base._method(self)
│   └── 协同初始化器：**kwargs 传递 ⭐
│       ├── 问题：super() 基于实例 MRO
│       ├── 错误方案：显式调用各基类初始化器
│       └── 正确方案：各初始化器接受 **kwargs 并向上传递
│
├── 七、混入（Mixins）⭐⭐
│   ├── 本质：组合 + 继承机制
│   ├── 特点：不完整类，依赖使用者提供属性
│   ├── SettingsFileMixin 示例
│   │   ├── .ini 配置文件 + configparser
│   │   ├── Greeter / MagicNumberPrinter 使用
│   │   └── 单一规范来源，所有使用者受益
│   └── 混入是"基于数据决策"规则的例外
│
└── 八、总结
    └── 谨慎使用继承，目标：可读、可维护的代码
```

- **警告与提示表**

| 类型   | 内容                                                         |
| ------ | ------------------------------------------------------------ |
| ⚠️ 警告 | 完全照搬 SOLID 字面规则仍可能写出糟糕代码，**必须运用常识**  |
| ⚠️ 警告 | 多重继承基类顺序**直接影响 MRO**，不一致顺序会导致 `TypeError` |
| ⚠️ 警告 | `super()` 基于**实例的** MRO（非当前类），协同初始化需用 `**kwargs` |
| ⚠️ 警告 | 显式调用 `Base.method(self)` 可绕过 MRO，但需**手动传 `self`** |
| ⚠️ 警告 | 上帝类（做太多事）和桩类（无数据）是继承的典型反模式         |
| ⚠️ 警告 | 混入**必须**由使用者提供缺失属性，否则运行时出错             |
| 💡 技巧 | 类的设计基于**组成数据**而非行为                             |
| 💡 技巧 | 鸭子类型优先于继承；纯接口需求用抽象基类                     |
| 💡 技巧 | 多重继承优先用协同初始化器（`**kwargs` 全量传递）            |
| 💡 技巧 | 混入是跨类共享方法的最佳方式之一，提供单一规范来源           |
| 💡 技巧 | `issubclass(sub, parent)` 检查继承关系；`ClassName.__mro__` 查看 MRO |